In [1]:
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
gemini_api_key = os.getenv("GEMINI_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if gemini_api_key:
    print(f"Gemini API Key exists and begins {gemini_api_key[:7]}")
else:
    print("Gemini API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"Openrouter API Key exists and begins {openrouter_api_key[:7]}")
else:
    print("Openrouter API Key not set (and this is optional)")

Anthropic API Key exists and begins sk-ant-
Gemini API Key exists and begins AQ.Ab8R
Openrouter API Key exists and begins sk-or-v


In [3]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

openai = OpenAI()

# For Gemini, DeepSeek and Groq, we can use the OpenAI python client
# Because Google and DeepSeek have endpoints compatible with OpenAI
# And OpenAI allows you to change the base_url

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
openrouter_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=gemini_api_key, base_url=gemini_url)
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [4]:
tell_a_joke = [
    {"role":"user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"}
]

In [ ]:
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

In [ ]:
response = gemini.chat.completions.create(model="gemini-2.5-pro", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

In [ ]:
response = anthropic.chat.completions.create(model="claude-sonnet-4-5-20250929", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

## Training vs Inference time scaling

In [5]:
easy_puzzle = [
    {
        "role": "user",
        "content": "you toss 2 coins. One of them is heads. What's the probability the other is tails? Answer with the probability only.",
    }
]

In [ ]:
response = openai.chat.completions.create(
    model="gpt-5-nano", messages=easy_puzzle, reasoning_effort="minimal"
)  # minimal, low, medium, high
display(Markdown(response.choices[0].message.content))

In [ ]:
response = openai.chat.completions.create(
    model="gpt-5-nano", messages=easy_puzzle, reasoning_effort="low"
)  # minimal, low, medium, high
display(Markdown(response.choices[0].message.content))

In [ ]:
response = openai.chat.completions.create(
    model="gpt-5-nano", messages=easy_puzzle, reasoning_effort="medium"
)  # minimal, low, medium, high
display(Markdown(response.choices[0].message.content))

In [ ]:
response = openai.chat.completions.create(
    model="gpt-5-nano", messages=easy_puzzle, reasoning_effort="high"
)  # minimal, low, medium, high
display(Markdown(response.choices[0].message.content))

## A Spicy challenge to test the competitive spirit

In [6]:
dilema_prompt = """
You and a partner are contestants on a game show. You're each taken to separate rooms and given a choice:
Cooperate: Choose "Share" — if both of you choose this, you each win $1,000.
Defect: Choose "Steal" — if one steals and the other shares, the stealer gets $2,000 and the sharer gets nothing.
If both steal, you both get nothing.
Do you choose to Steal or Share? Pick one.
"""

dilema = [{"role": "user", "content": dilema_prompt}]

In [ ]:
response = anthropic.chat.completions.create(
    model="claude-sonnet-4-5-20250929", messages=dilema
)
display(Markdown(response.choices[0].message.content))

In [ ]:
response = openai.chat.completions.create(
    model="claude-sonnet-4-5-20250929", messages=dilema
)
display(Markdown(response.choices[0].message.content))

### Local Models with OLLAMA

In [ ]:
requests.get("http://localhost:11434").content

In [ ]:
!ollama pull llama3.2

<!-- !ollama pull gpt-oss:20b -->

In [ ]:
response = ollama.chat.completions.create(model="llama3.2",messages=easy_puzzle)
display(Markdown(response.choices[0].message.content))

In [ ]:
response = openrouter.chat.completions.create(model="z-ai/glm-4.5", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

### And now first look at the powerful, mighty (and quite heavyweight) LangChain

In [ ]:
from langchain_openai import ChatOpenAI

In [ ]:
llm = ChatOpenAI(model="gpt-5-mini")
response = llm.invoke(tell_a_joke)

display(Markdown(response.content))

### Finally - the wonderful lightweight LiteLLM

In [7]:
from litellm import completion

response = completion(model="openai/gpt-5-mini", messages=tell_a_joke)

reply = response.choices[0].message.content
display(Markdown(reply))

- Favorite exercise for an LLM engineering student? Gradient descent — it's all about losing weights.

- Student: "How long until I'm an LLM expert?" Mentor: "About three coffees, two all-nighters, and one lucky random seed."

- How many LLM engineers does it take to change a lightbulb? One to change it, one to fine-tune a replacement, and a whole team to debate whether the new bulb is aligned.

In [9]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output Tokens: {response.usage.completion_tokens}")
print(f"Total Tokens:{response.usage.total_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 23
Output Tokens: 1125
Total Tokens:1148
Total cost: 0.2256 cents


In [10]:
with open("hamlet.txt",mode='r', encoding="utf-8") as f:
    hamlet = f.read()

In [16]:
loc = hamlet.find("Speak, man")
print(hamlet[loc:loc+100])

Speak, man.
  Laer. Where is my father?
  King. Dead.
  Queen. But not by him!
  King. Let him deman


In [18]:
question = [
    {
        "role": "user",
        "content": "In Hamlet, when Laertes asks ' Where is my father?' What is the reply?",
    }
]

In [20]:
response = completion(model="gemini/gemini-2.5-flash-lite",messages=question)
display(Markdown(response.choices[0].message.content))


In Shakespeare's *Hamlet*, when Laertes asks "Where is my father?" (Act IV, Scene V), the reply comes from **Gertrude**, Hamlet's mother and Claudius's wife.

She tells him:

> "One woe doth tread upon another's heel,
> So fast they follow. Your sister's drowned, Laertes."

She then goes on to explain the circumstances of Ophelia's death, which Laertes is deeply distraught about. She does not directly answer "where" his father (Polonius) is, because he is already dead, and she is focused on delivering the devastating news about Ophelia.

In [21]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output Tokens: {response.usage.completion_tokens}")
print(f"Total Tokens:{response.usage.total_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 19
Output Tokens: 137
Total Tokens:156
Total cost: 0.0057 cents


In [27]:
question[0]["content"] +="\n\n For context, here is the entire text of Hamlet:\n\n" + hamlet

In [28]:
response = completion(model="gemini/gemini-2.5-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))

When Laertes asks "Where is my father?", the reply comes from **King Claudius**:

"Dead."

In [29]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output Tokens: {response.usage.completion_tokens}")
print(f"Total Tokens:{response.usage.total_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 53208
Output Tokens: 23
Total Tokens:53231
Total cost: 0.5330 cents


In [30]:
response = completion(model="gemini/gemini-2.5-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))

When Laertes asks, "Where is my father?", the reply comes from Claudius:

"**Dead.**"

In [31]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output Tokens: {response.usage.completion_tokens}")
print(f"Total Tokens:{response.usage.total_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 53208
Output Tokens: 24
Total Tokens:53232
Total cost: 0.0631 cents


In [56]:
gpt_model = "gpt-4.1-mini"
# claude_model = "claude-3-5-haiku-latest"
claude_model = "claude-sonnet-4-5-20250929"

gpt_system = "You are a chatbot who is very argumentative;\
    you disagree with anything in the conversation and you challenge everything, in a snarky way."

claude_system = "You are a very polite, courteous chatbot. you try to agree with\
    everything the other person is says, or find common ground. If the other person is argumentative,\
        you try to calm them doen and keep chatting."

gpt_messages = ["Hi there"]
claude_messages = ["Hi"]

In [47]:
def call_gpt():
    messages = [{"role": "system", "content": gpt_system}]
    for gpt, claude in zip(gpt_messages,claude_messages):
        messages.append({"role": "assistant", "content": gpt})
        messages.append({"role": "user", "content": claude})
    response = openai.chat.completions.create(model=gpt_model, messages=messages)

    return response.choices[0].message.content

In [48]:
call_gpt()

"Oh, wow, what an original greeting. Haven't heard that one a million times before. What’s next, are we just going to stare at each other in awkward silence now?"

In [54]:
def call_claude():
    messages = [{"role": "system", "content": claude_system}]
    for gpt, claude in zip(gpt_messages, claude_messages):
        messages.append({"role": "user", "content": gpt})
        messages.append({"role": "assistant", "content": claude})
    messages.append({"role": "user", "content": gpt_messages[-1]})
    response = anthropic.chat.completions.create(model=claude_model, messages=messages)

    return response.choices[0].message.content

In [57]:
call_claude()

"Hello! How are you doing today? It's nice to hear from you! 😊 \n\nIs there anything on your mind that you'd like to chat about? I'm here and happy to talk about whatever interests you!"